# `train.py` — Main Training Script

## Purpose
The end-to-end training pipeline for the `MultiAspectSentimentModel`. This is the script you run to train **a single model** from a YAML config (as opposed to `experiment_runner.py` which runs multiple experiments back-to-back).

## Usage
```bash
# From ml-research root:
python src/models/train.py --config configs/config.yaml

# Resume from a crash:
python src/models/train.py --config configs/config.yaml --resume results/exp_name/checkpoint_epoch_10.pt
```

## Training Pipeline
```
1.  Load config.yaml
2.  Set random seeds (reproducibility)
3.  Init tokenizer + (optional) dependency parser
4.  Create CosmeticReviewDataset → train/val/test DataLoaders
5.  Build MultiAspectSentimentModel
6.  Compute per-aspect class weights → AspectSpecificLossManager
7.  Build AdamW optimizer + linear warmup+decay scheduler
8.  For each epoch:
      a. train_epoch(): forward + AspectSpecificLoss.compute_loss() + backward + clip_grad
      b. evaluate(val_loader): compute Macro-F1
      c. Save best_model.pt if val improves
      d. Early stopping if patience exceeded
9.  Load best_model.pt → evaluate on test set
10. Run MixedSentimentEvaluator on test set
11. Save test_results.json
```

## Logging
- **TensorBoard**: `runs/{exp_name}/` — scalars for train loss, val Macro-F1, LR
- **Weights & Biases**: optional, enabled by `experiment.use_wandb: true` in config

In [ ]:
import os, yaml, torch, torch.nn as nn, argparse, sys
from torch.cuda.amp import autocast, GradScaler
from transformers     import RobertaTokenizer, get_linear_schedule_with_warmup
import numpy as np
from tqdm import tqdm
from pathlib import Path

ML_RESEARCH = os.path.dirname(os.path.abspath(''))  # ml-research/
SRC_DIR = os.path.join(ML_RESEARCH, 'src')
EXP_DIR = os.path.join(SRC_DIR, 'experiments')
for _p in [SRC_DIR, EXP_DIR]:
    if _p not in sys.path: sys.path.insert(0, _p)

from models.model    import create_model
from models.losses   import AspectSpecificLossManager
from utils.data_utils import create_dataloaders, DependencyParser, compute_class_weights
from utils.metrics   import AspectSentimentEvaluator, MixedSentimentEvaluator

## Trainer Class
All training state is encapsulated in `Trainer`. Instantiate it with a config path, then call `.train()`.

In [ ]:
class Trainer:
    def __init__(self, config_path: str):
        with open(config_path) as f:
            self.config = yaml.safe_load(f)

        self._set_seed(self.config['experiment']['seed'])  # Reproducibility

        self.device  = torch.device(self.config['hardware']['device']
                                    if torch.cuda.is_available() else 'cpu')
        self.save_dir = Path(self.config['experiment']['save_dir']) / self.config['experiment']['name']
        self.save_dir.mkdir(parents=True, exist_ok=True)

        self.tokenizer         = RobertaTokenizer.from_pretrained(self.config['model']['roberta_model'])
        self.dependency_parser = None

        # Optionally load spaCy parser for GCN variants
        if self.config['data'].get('use_dependency_parsing', False):
            self.dependency_parser = DependencyParser()

        self.train_loader, self.val_loader, self.test_loader = create_dataloaders(
            self.config, self.tokenizer, self.dependency_parser
        )

        self.model = create_model(self.config)
        self.model.to(self.device)

        # Compute class counts per aspect → builds per-aspect HybridLoss instances
        aspect_class_counts = compute_class_weights(
            self.config['data']['train_path'],
            self.config['aspects']['names'],
            self.config['aspects']['label_map']
        )
        self.loss_manager = AspectSpecificLossManager(aspect_class_counts, self.config['training'])

        # AdamW: weight decay regularises non-bias transformer layers
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.config['training']['learning_rate'],
            weight_decay=self.config['training']['weight_decay']
        )

        # Linear warmup + linear decay LR schedule (standard for transformer fine-tuning)
        num_training_steps = len(self.train_loader) * self.config['training']['num_epochs']
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=self.config['training']['warmup_steps'],
            num_training_steps=num_training_steps
        )

        # AMP (FP16) — reduces GPU memory and speeds up training;
        # GradScaler handles gradient underflow from FP16 precision.
        self.use_amp = self.config['hardware'].get('mixed_precision', False)
        if self.use_amp:
            self.scaler = GradScaler()

        # TensorBoard logging
        from torch.utils.tensorboard import SummaryWriter
        self.writer = SummaryWriter(log_dir=self.save_dir, flush_secs=30)

        self.evaluator       = AspectSentimentEvaluator(self.config['aspects']['names'])
        self.global_step     = 0
        self.best_val_metric = 0
        self.patience_counter = 0

    @staticmethod
    def _set_seed(seed):
        """Set all random seeds for reproducible training."""
        import random
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        np.random.seed(seed); random.seed(seed)
        torch.backends.cudnn.deterministic = True

    def _forward(self, batch):
        """Run forward pass; attaches edge_indices for GCN if enabled."""
        input_ids      = batch['input_ids'].to(self.device)
        attention_mask = batch['attention_mask'].to(self.device)
        aspect_ids     = batch['aspect_ids'].to(self.device)
        edge_indices   = None
        if self.config['model'].get('use_dependency_gcn', False):
            edge_indices = [e.to(self.device) if e is not None else None
                            for e in batch['edge_indices']]
        return self.model(input_ids, attention_mask, aspect_ids, edge_indices)

    def train_epoch(self, epoch: int) -> float:
        """Full training pass over one epoch. Logs to TensorBoard. Returns avg loss."""
        self.model.train()
        total_loss = 0

        for batch_idx, batch in enumerate(tqdm(self.train_loader, desc=f'Epoch {epoch+1}')):
            labels     = batch['labels'].to(self.device)
            aspect_ids = batch['aspect_ids'].to(self.device)
            self.optimizer.zero_grad()

            if self.use_amp:
                with autocast():
                    preds = self._forward(batch)
                    if isinstance(preds, tuple): preds = preds[0]
                    loss, _ = self.loss_manager.compute_loss(preds, labels, aspect_ids, self.config['aspects']['names'])
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['training']['max_grad_norm'])
                self.scaler.step(self.optimizer); self.scaler.update()
            else:
                preds = self._forward(batch)
                if isinstance(preds, tuple): preds = preds[0]
                loss, _ = self.loss_manager.compute_loss(preds, labels, aspect_ids, self.config['aspects']['names'])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['training']['max_grad_norm'])
                self.optimizer.step()

            self.scheduler.step()
            total_loss += loss.item()

            if self.global_step % self.config['experiment']['log_every'] == 0:
                avg = total_loss / (batch_idx + 1)
                self.writer.add_scalar('train/loss', loss.item(), self.global_step)
                self.writer.add_scalar('train/avg_loss', avg, self.global_step)
                self.writer.add_scalar('train/lr', self.scheduler.get_last_lr()[0], self.global_step)

            self.global_step += 1

        return total_loss / len(self.train_loader)

    def evaluate(self, dataloader, split_name='Validation') -> dict:
        """Compute Accuracy, Macro-F1, MCC for each aspect and overall on the given DataLoader."""
        self.model.eval()
        all_preds, all_labels, all_aspects = [], [], []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f'Evaluating {split_name}'):
                preds = self._forward(batch)
                if isinstance(preds, tuple): preds = preds[0]
                all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
                all_labels.extend(batch['labels'].numpy())
                all_aspects.extend(batch['aspects'])

        aspect_metrics = {
            asp: self.evaluator.evaluate_aspect(
                np.array(all_labels)[np.array([a == asp for a in all_aspects])],
                np.array(all_preds )[np.array([a == asp for a in all_aspects])],
                asp
            )
            for asp in self.config['aspects']['names']
        }
        overall = self.evaluator.evaluate_aspect(np.array(all_labels), np.array(all_preds), 'overall')
        print(f'{split_name} — Macro-F1: {overall["macro_f1"]:.4f}  Acc: {overall["accuracy"]:.4f}')
        return {'overall': overall, 'aspects': aspect_metrics}

    def save_checkpoint(self, filename):
        torch.save({
            'model_state_dict'    : self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'global_step'         : self.global_step,
            'best_val_metric'     : self.best_val_metric,
            'config'              : self.config,
        }, self.save_dir / filename)

    def load_checkpoint(self, path):
        """Resume training from a checkpoint (restores optimizer + scheduler state too)."""
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state_dict'])
        self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        self.scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        self.global_step      = ckpt['global_step']
        self.best_val_metric  = ckpt['best_val_metric']

    def train(self) -> dict:
        """Full training loop with early stopping, checkpoint saving, and post-train test evaluation."""
        for epoch in range(self.config['training']['num_epochs']):
            train_loss = self.train_epoch(epoch)
            print(f'Epoch {epoch+1} — Train Loss: {train_loss:.4f}')

            val_metrics = self.evaluate(self.val_loader, 'Validation')
            val_f1      = val_metrics['overall']['macro_f1']

            self.writer.add_scalar('val/macro_f1',  val_f1, self.global_step)
            self.writer.add_scalar('val/accuracy', val_metrics['overall']['accuracy'], self.global_step)

            if val_f1 > self.best_val_metric:
                self.best_val_metric  = val_f1
                self.patience_counter = 0
                self.save_checkpoint('best_model.pt')  # Save whenever val improves
                print(f'  → New best! Val Macro-F1: {val_f1:.4f}')
            else:
                self.patience_counter += 1
                print(f'  Patience: {self.patience_counter}/{self.config["training"]["early_stopping_patience"]}')
                if self.patience_counter >= self.config['training']['early_stopping_patience']:
                    print(f'Early stopping after epoch {epoch+1}'); break

            # Periodic checkpoint every 5 epochs (not just best)
            if (epoch + 1) % 5 == 0:
                self.save_checkpoint(f'checkpoint_epoch_{epoch+1}.pt')

        # ── Final test evaluation ──────────────────────────────────────────────
        best_ckpt = self.save_dir / 'best_model.pt'
        if best_ckpt.exists(): self.load_checkpoint(best_ckpt)
        test_metrics = self.evaluate(self.test_loader, 'Test')

        # ── MSR Evaluation ────────────────────────────────────────────────────
        msr        = MixedSentimentEvaluator(self.config['aspects']['names'])
        rev_true, rev_pred = {}, {}
        self.model.eval()
        with torch.no_grad():
            for batch in self.test_loader:
                preds   = self._forward(batch)
                if isinstance(preds, tuple): preds = preds[0]
                classes = torch.argmax(preds, dim=1).cpu().numpy()
                for i in range(len(classes)):
                    rid = batch['review_ids'][i]; asp = batch['aspects'][i]
                    rev_true.setdefault(rid, {})[asp] = batch['labels'][i].item()
                    rev_pred.setdefault(rid, {})[asp] = int(classes[i])

        mixed_metrics = msr.evaluate_mixed_sentiment_resolution(rev_true, rev_pred)
        msr.print_mixed_sentiment_results(mixed_metrics)
        test_metrics['mixed_sentiment'] = mixed_metrics

        # Save test_results.json
        import json
        def _ser(obj):
            if isinstance(obj, np.ndarray): return obj.tolist()
            if isinstance(obj, (np.int64, np.int32)): return int(obj)
            if isinstance(obj, (np.float64, np.float32)): return float(obj)
            if isinstance(obj, dict): return {k: _ser(v) for k, v in obj.items()}
            if isinstance(obj, list): return [_ser(x) for x in obj]
            return obj
        (self.save_dir / 'test_results.json').write_text(
            json.dumps(_ser(test_metrics), indent=2), encoding='utf-8'
        )
        self.writer.close()
        return test_metrics

## Run Training

In [ ]:
# ── To train interactively: ────────────────────────────────────────────────────
# trainer = Trainer('configs/config.yaml')
# trainer.train()

# ── To resume from crash: ──────────────────────────────────────────────────────
# trainer = Trainer('configs/config.yaml')
# trainer.load_checkpoint('results/my_experiment/checkpoint_epoch_10.pt')
# trainer.train()

print('train.py loaded. Instantiate Trainer(config_path) and call .train() to start training.')